# 3 — How many households had a measurable intervention?

**And how big was it?**

Source: `dose_response_recon.py`.

> ⚠️ **The outputs below are already saved — just scroll and read.**
> Do **not** press *Run*. This reads the 5 GB HEAPO dataset from a local folder
> that is not attached here, so re-running produces only `FileNotFoundError`.
> Everything you need to see is stored in the cells.

**Headline: 12 of 89** households have a documented heating curve reduction of ≥1 K.
That is the ceiling on any dose-response analysis of the most common intervention.


In [1]:
import dose_response_recon as D

pr  = D.load("protocols")
q15 = D.load("15min")
print(f"protocols {pr.shape[0]} rows, 15min overview {q15.shape[0]} rows")

protocols 410 rows, 15min overview 1407 rows


## Base — rebuilding the 89

Includes the finding that `DaysBeforeVisit` counts days *with data*, not calendar
days, so the visit date cannot be reconstructed by arithmetic.

In [2]:
u = D.step0_base(pr, q15)

BASE -- protocol households with >= 180 d before AND after (15min overview)
linkable protocol rows: 217   distinct households: 214
rows meeting the rule: 89   distinct households: 89
CONFIRMED: 89 households, matching recon.py.

repeat-visit handling:
  households with >1 protocol visit overall: 3 -> [np.int64(111119), np.int64(120912), np.int64(8087988)]
  of those, inside the usable sample: 0 -> []
  => NONE of the repeat-visit households survives the >= 180 d both-sides filter, so
     'first visit', 'last visit' and 'exclude' all yield the SAME sample. Counts both
     ways: first=89  last=89  excluded=89.
     The configured policy (last) is therefore a no-op here; it is
     still applied below so the script stays correct if the filter is ever loosened.
  sample after applying policy: 89 rows / 89 households

--- inspection findings before proceeding ---
1. Visit_Date is present for 89/89 of the sample -- unlike the
   full protocol file (304/410), every usable household has a re

## Step 1 — Heating curve reduction

In [3]:
curve = D.step1_curve(u)

STEP 1 -- HEATING CURVE REDUCTION WITHIN THE 89
Reduction is defined as Before - After, so a positive number is a reduction in supply
temperature (the intended direction of the intervention).

--- Outside20 ---
  both Before and After present: 13 of 89 (14.6%)   -> missing for 76
  non-zero change: 5
  reduction (After < Before): 4    increase: 1
  reduction magnitude, Kelvin (all observed deltas, including zeros):
    count=13  mean=0.73 K  median=0.00 K  min=-1.50 K  max=5.00 K
    q25=0.00 K  q50=0.00 K  q75=1.00 K
  histogram, 1 K bins:
    [-2, -1) K: 1  #
    [+0, +1) K: 8  ########
    [+1, +2) K: 1  #
    [+2, +3) K: 1  #
    [+3, +4) K: 1  #
    [+5, +6) K: 1  #

--- Outside0 ---
  both Before and After present: 11 of 89 (12.4%)   -> missing for 78
  non-zero change: 7
  reduction (After < Before): 7    increase: 0
  reduction magnitude, Kelvin (all observed deltas, including zeros):
    count=11  mean=3.00 K  median=2.00 K  min=0.00 K  max=8.00 K
    q25=0.00 K  q50=2.00 K  q

## Step 2 — Other interventions

In [4]:
other = D.step2_other(u)

STEP 2 -- OTHER INTERVENTIONS WITHIN THE 89
--- heating limit ---
  both values present: 17 of 89
  changed: 14   reduced: 14   raised: 0
  magnitude, degrees C:
    count=17  mean=1.53 C  median=2.00 C  min=0.00 C  max=3.00 C
    q25=1.00 C  q50=2.00 C  q75=2.00 C
  histogram, 1 C bins:
    [+0, +1) C: 3  ###
    [+1, +2) C: 4  ####
    [+2, +3) C: 8  ########
    [+3, +4) C: 2  ##

--- night setback ---
  activated before visit: 30
  GENUINE DEACTIVATION (before True, after False): 21
  still active after visit: 9
  after-value missing where before was True: 0
  switched ON at the visit (before False, after True): 1

--- DHW temperature ---
  both values present: 10 of 89
  changed: 10   reduced: 8   raised: 2
  magnitude, Kelvin:
    count=10  mean=2.40 K  median=3.00 K  min=-5.00 K  max=10.00 K
    q25=2.25 K  q50=3.00 K  q75=3.75 K
  histogram, 1 K bins:
    [-5, -4) K: 2  ##
    [+2, +3) K: 1  #
    [+3, +4) K: 4  ####
    [+4, +5) K: 1  #
    [+6, +7) K: 1  #
    [+10, +11) K: 1

## Step 3 — Intervention profile per household

In [5]:
prof = curve.merge(other, on="Household_ID")
prof = D.step3_profile(prof)

STEP 3 -- INTERVENTION PROFILE PER HOUSEHOLD
NOTE ON SCOPE: the brief says 'the four interventions'. STEP 1 defines one (heating curve)
and STEP 2 defines four more (heating limit, night setback, DHW temperature, circulation
pump), so FIVE are profiled here. Dropping any of them would hide a real intervention;
the counts below are per-intervention, so a four-way reading is still recoverable.

interventions recorded per household (n=89):
  exactly 0:  41 households (46.1%)
  exactly 1:  35 households (39.3%)
  exactly 2:  10 households (11.2%)
  exactly 3:   3 households (3.4%)
  exactly 4:   0 households (0.0%)
  exactly 5:   0 households (0.0%)
  mean interventions per household: 0.72   max: 3
  at least one: 48   none at all: 41

per-intervention totals:
  heating curve >=1K      12
  heating limit           14
  night setback off       21
  DHW temperature         10
  circulation pump         7

combinations occurring 3+ times:
   41  (none)
   12  night setback off
    9  heating 

## Step 4 — Data sufficiency per intervention

In [6]:
D.step4_sufficiency(u, prof)

STEP 4 -- DATA SUFFICIENCY PER INTERVENTION TYPE
*** The winter columns are a COVERAGE UPPER BOUND derived from the earliest/latest
*** timestamps and the visit date only. A season counts when it lies entirely inside the
*** relevant span. It does NOT verify that the days inside carry measurements -- and the
*** gaps measured in the BASE section (median 5 d, max 317 d) prove they often do not.
*** Real usable winters can only be fewer.

whole sample (n=89):
  >= 180 d each side:  89
  >= 270 d each side:  81
  >= 365 d each side:  68
  >= 1 complete winter each side: 38

      intervention  affected  >=180d both  >=270d both  >=365d both  >=1 winter both
heating curve >=1K        12           12           12            9                3
     heating limit        14           14           14           14                5
 night setback off        21           21           19           17               14
   DHW temperature        10           10            9            8               

## Step 5 — Verdict

In [7]:
headline = int(prof["curve_reduced_1K"].astype(bool).sum())
D.step5_verdict(prof, headline, len(u))

STEP 5 -- VERDICT
Thresholds (stated so they can be argued with), counted as affected households:
  (a) dose-response with a continuous magnitude : >= 25
      Rationale: a continuous dose needs enough spread to fit and check a slope; below
      ~25 points a single household moves the estimate.
  (b) binary treated/untreated comparison only  : >= 15
  (c) descriptive case series only              : below 15

      intervention  affected  isolated (attributable)                     verdict
heating curve >=1K        12                        9 (c) descriptive case series
     heating limit        14                        7 (c) descriptive case series
 night setback off        21                       12  (b) binary comparison only
   DHW temperature        10                        4 (c) descriptive case series
  circulation pump         7                        3 (c) descriptive case series

summary: (a) 0   (b) 1   (c) 4

**************************************************************

## What this establishes

- **12 of 89** have a heating curve reduction ≥1 K. Moving the threshold barely moves
  the number — the limit is what was *recorded*, not how big the changes were.
- **46% of the 89 had no recorded intervention at all.**
- Interventions are mostly **separable** (only 27% of affected households got two or
  more), so confounding is not the problem — group size is.
- **No intervention type supports dose-response analysis.** Night setback (21 cases)
  supports a binary comparison; everything else is a case series.
